## AI-Powered Sales Forecasting & Business Intelligence Dashboard
### Feature Engineering Notebook

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
## Load Cleaned Dataset
df = pd.read_csv("../data/processed/cleaned_superstore.csv")

print(df.shape)
df.head()

(9994, 20)


,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### Create Time-Based Features

In [3]:
## Convert Date Columns
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

In [4]:
## Year
df["order_year"] = df["Order Date"].dt.year

## Month
df["order_month"] = df["Order Date"].dt.month

## Day
df["order_day"] = df["Order Date"].dt.day

## Quarter
df["order_quarter"] = df["Order Date"].dt.quarter

## Day of week
df["order_dayofweek"] = df["Order Date"].dt.dayofweek

## Week of year
df["order_weekofyear"] = df["Order Date"].dt.isocalendar().week

### Shipping Features
**Shipping Duration**

Business Value:
Helps analyze delivery efficiency.

In [5]:
df['shipping_days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['shipping_days'].head()

0    3
1    3
2    4
3    7
4    7
Name: shipping_days, dtype: int64

### Profitability Features
**Profit Margin**

Business Value:
Measures profitability efficiency.

In [6]:
df['profit_margin'] = (
    df['Profit']/df['Sales']
)*100

df['profit_margin'].head()

0    16.00
1    30.00
2    47.00
3   -40.00
4    11.25
Name: profit_margin, dtype: float64

### Discount Features
**High Discount Indicator**
* 1 for high discount (greater than 20%)
* else 0

Business Value:
Detect heavily discounted transactions.

In [7]:
df["high_discount"] = np.where(
    df["Discount"] > 0.2,
    1,
    0
)

df["high_discount"].head()

0    0
1    0
2    0
3    1
4    0
Name: high_discount, dtype: int64

### Sales Performance Features
**Sales Per Quantity**

Business Value:
Measures average sales value per unit.

In [8]:
df['sales_per_quantity'] = (
    df['Sales'] / df['Quantity']
)

df['sales_per_quantity'].head()

0    130.9800
1    243.9800
2      7.3100
3    191.5155
4     11.1840
Name: sales_per_quantity, dtype: float64

### Customer Features
**Unique Customer Frequency**

Business Value:
Identify repeat customers.

In [9]:
customer_orders = df.groupby("Customer ID")["Order ID"].transform("count")

df["customer_order_frequency"] = customer_orders

In [10]:
df["customer_order_frequency"].head()

0     5
1     5
2     9
3    15
4    15
Name: customer_order_frequency, dtype: int64

### Regional Sales Features
**Average Regional Sales**

Business Value:
Understand regional performance patterns.

In [11]:
regional_avg_sales = (
    df.groupby("Region")["Sales"]
    .transform("mean")
)

df["regional_avg_sales"] = regional_avg_sales
df["regional_avg_sales"].head()

0    241.803645
1    241.803645
2    226.493233
3    241.803645
4    241.803645
Name: regional_avg_sales, dtype: float64

### Product-Level Features
**Product Sales Frequency**

Business Value:
Identify frequently sold products.

In [12]:
product_frequency = (
    df.groupby("Product ID")["Order ID"]
    .transform("count")
)

df["product_order_frequency"] = product_frequency

### Encoding
Label Encoding for Category

In [13]:
label_encoder = LabelEncoder()

df["category_encoded"] = label_encoder.fit_transform(
    df["Category"]
)
df["category_encoded"].head()

0    0
1    0
2    1
3    0
4    1
Name: category_encoded, dtype: int64

Segment Encoding

In [14]:
df["segment_encoded"] = label_encoder.fit_transform(
    df["Segment"]
)

### Forecasting-Oriented Features
**Lag Feature** (Previous Sales)

Business Value:
Very important for forecasting models.

In [15]:
df = df.sort_values("Order Date")
df["previous_sales"] = df["Sales"].shift(1)

df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,...,shipping_days,profit_margin,high_discount,sales_per_quantity,customer_order_frequency,regional_avg_sales,product_order_frequency,category_encoded,segment_encoded,previous_sales
7980,CA-2014-103800,2014-01-03,2014-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,...,4,33.75,0,8.224,17,215.772661,6,1,0,NaN
739,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,4,36.25,0,3.928,10,215.772661,6,1,2,16.448
740,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,4,-23.75,0,90.912,10,215.772661,10,1,2,11.784
741,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,4,-155.00,1,1.770,10,215.772661,5,1,2,272.736
1759,CA-2014-141817,2014-01-05,2014-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,...,7,25.00,0,6.512,10,238.336110,6,1,0,3.540


In [16]:
df[['Sales', 'previous_sales']].head()

,Sales,previous_sales
7980,16.448,NaN
739,11.784,16.448
740,272.736,11.784
741,3.540,272.736
1759,19.536,3.540


**Rolling Mean Sales**

Business Value:
Captures moving sales trends.

In [17]:
df["rolling_sales_mean"] = (
    df["Sales"]
    .rolling(window=7)
    .mean()
)

df[["Sales", "rolling_sales_mean"]].head(20)

,Sales,rolling_sales_mean
7980,16.448,NaN
739,11.784,NaN
740,272.736,NaN
741,3.540,NaN
1759,19.536,NaN
7476,5.480,NaN
7474,2573.820,414.763429
7475,609.980,499.553714
7180,12.780,499.696000
7477,391.980,516.730857


### Handle Newly Generated Missing Values

In [18]:
df.isna().sum()

Order ID                    0
Order Date                  0
Ship Date                   0
Ship Mode                   0
Customer ID                 0
Customer Name               0
Segment                     0
Country                     0
City                        0
State                       0
Postal Code                 0
Region                      0
Product ID                  0
Category                    0
Sub-Category                0
Product Name                0
Sales                       0
Quantity                    0
Discount                    0
Profit                      0
order_year                  0
order_month                 0
order_day                   0
order_quarter               0
order_dayofweek             0
order_weekofyear            0
shipping_days               0
profit_margin               0
high_discount               0
sales_per_quantity          0
customer_order_frequency    0
regional_avg_sales          0
product_order_frequency     0
category_e

In [19]:
df = df.fillna(0)

df.isna().sum()

Order ID                    0
Order Date                  0
Ship Date                   0
Ship Mode                   0
Customer ID                 0
Customer Name               0
Segment                     0
Country                     0
City                        0
State                       0
Postal Code                 0
Region                      0
Product ID                  0
Category                    0
Sub-Category                0
Product Name                0
Sales                       0
Quantity                    0
Discount                    0
Profit                      0
order_year                  0
order_month                 0
order_day                   0
order_quarter               0
order_dayofweek             0
order_weekofyear            0
shipping_days               0
profit_margin               0
high_discount               0
sales_per_quantity          0
customer_order_frequency    0
regional_avg_sales          0
product_order_frequency     0
category_e

### Final Dataset Overview

In [19]:
df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,...,profit_margin,high_discount,sales_per_quantity,customer_order_frequency,regional_avg_sales,product_order_frequency,category_encoded,segment_encoded,previous_sales,rolling_sales_mean
7980,CA-2014-103800,2014-01-03,2014-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,...,33.75,0,8.224,17,215.772661,6,1,0,NaN,NaN
739,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,36.25,0,3.928,10,215.772661,6,1,2,16.448,NaN
740,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,-23.75,0,90.912,10,215.772661,10,1,2,11.784,NaN
741,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,...,-155.00,1,1.770,10,215.772661,5,1,2,272.736,NaN
1759,CA-2014-141817,2014-01-05,2014-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,...,25.00,0,6.512,10,238.336110,6,1,0,3.540,NaN


In [20]:
df.shape

(9994, 37)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9994 entries, 7980 to 906
Data columns (total 37 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Order ID                  9994 non-null   object        
 1   Order Date                9994 non-null   datetime64[ns]
 2   Ship Date                 9994 non-null   datetime64[ns]
 3   Ship Mode                 9994 non-null   object        
 4   Customer ID               9994 non-null   object        
 5   Customer Name             9994 non-null   object        
 6   Segment                   9994 non-null   object        
 7   Country                   9994 non-null   object        
 8   City                      9994 non-null   object        
 9   State                     9994 non-null   object        
 10  Postal Code               9994 non-null   int64         
 11  Region                    9994 non-null   object        
 12  Product ID             

### Save Feature-Engineered Dataset

In [23]:
df.to_csv(
    "../data/processed/feature_engineered_superstore.csv",
    index=False
)